In [ ]:
import sys
import numpy as np
import pandas as pd
import yaml
import matplotlib as mpl
import matplotlib.pyplot as plt

sys.path.insert(0, 'aoeb/e2e')                         # e2e helpers (calibration.machine: aoeb/config/budget_wfsw.yml on the WFSW products)
import utils as e2e
from specula.lib.n_phot import n_phot
from aoeb.chromatic import ChromaticModel
from aoeb.psf import PsfEngine                         # separate from the budget: budget.py does not import psf.py
from aoeb import plotting as pl

plt.rcParams['image.cmap'] = 'inferno'
plt.rcParams['image.origin'] = 'lower'
plt.rcParams.update({"text.usetex": True, "font.family": "serif", "font.serif": ["Computer Modern Roman"]})

In [ ]:
# ---- System of aoeb/e2e/ristretto_main.yml: every parameter below comes from that file
YML = 'aoeb/e2e/ristretto_main.yml'
with open(YML) as f:
    Y = yaml.safe_load(f)

m = e2e.machine()                 # ErrorBudgetMachine: root /calibration/WFSW, tags and sensors of budget_wfsw.yml, atmosphere of ristretto_main
det = m.detector
CONTROLLER = 'leaky'              # aoeb has int | leaky | wiener | IirControllers: no Kalman, the e2e loop fuses both pyramids with one
dt = Y['main']['time_step']
zenith = Y['main']['zenithAngleInDeg']
seeing = Y['seeing']['constant'] / np.cos(np.radians(zenith)) ** 0.6     # along the line of sight (r0 ~ airmass^-3/5)
latency_s = Y['kpred']['delay'] * dt                                     # fractional delay of kpred: 0.5 step = 250 us
N_dm = Y['dm']['nmodes']
area = (m.pupil() > 0).sum() * Y['main']['pixel_pitch'] ** 2            # [m^2] pupil area seen by SPECULA

def sensor(name):
    """Single-stage loop of pyr_<name> (its source, camera and the science source of the same band)."""
    pyr, ccd, src, sci = Y[f'pyr_{name}'], Y[f'cred_{name}'], Y[f'source_{name}'], Y[f'science_{name}']
    assert pyr['wavelengthInNm'] == src['wavelengthInNm']
    assert (ccd['readout_level'], ccd['quantum_eff'], ccd['excess_noise']) == (det.ron_e, det.qe, det.excess_noise), 'detector != budget_wfsw.yml'
    n = int(pyr['pup_diam'])
    flux = n_phot(src['magnitude'], lambda_=src['wavelengthInNm'] * 1e-9, width=1e-9)[0] * ccd['bandw'] * area   # [ph/s], as SPECULA's CCD
    return dict(rMod=float(pyr['mod_amp']), n_subap=n, wavelength_nm=float(pyr['wavelengthInNm']), fs=1.0 / ccd['dt'],
                N=min(N_dm, m.max_modes(n)), magnitude=src['magnitude'], flux=flux, science_nm=float(sci['wavelengthInNm']))

S = {k: sensor(k) for k in ('vis', 'nir')}
atm = Y['atmo']
assert m.L0 == atm['L0']
chrom = {k: ChromaticModel(atm['heights'], atm['Cn2'], science_nm=s['science_nm'], zenith_deg=zenith,
                           multiplicative=False, fresnel=False, scintillation=False)                 # optional terms off (default)
         for k, s in S.items()}

print(f"seeing {Y['seeing']['constant']}\" at z = {zenith} deg -> {seeing:.3f}\" on the line of sight (products of "
      f"{e2e.snap_seeing(seeing)}\"), L0 {m.L0} m, effective wind {m.wind:.2f} m/s, latency {latency_s * 1e6:.0f} us, {det}")
pd.DataFrame(S)

In [ ]:
# Budget of each pyramid as a single-stage loop (requested fs and N: no frame-rate optimisation)
R = {k: m.budget(rMod=s['rMod'], n_subap=s['n_subap'], seeing=seeing, flux=s['flux'], fs=s['fs'], N=s['N'], latency_s=latency_s,
                 controller=CONTROLLER, scenario='standard', wavelength_nm=s['wavelength_nm'], crosstalk=True, chromatic=chrom[k],
                 optimize_freq=False)
     for k, s in S.items()}
# lag, noise (+ noise_photon, noise_ron), alias, xtalk, fitting, chrom_shear, total [nm rms]; SR at the science wavelength of the band
pd.DataFrame({k: {**r.totals(), 'SR_science': r.strehl(S[k]['science_nm']), 'NmodesUsed': r.params['NmodesUsed'],
                  'freqUsedInHz': r.params['freqUsedInHz']} for k, r in R.items()}).round(4)

In [ ]:
# Calibration checks (products at the seeings of budget_wfsw.yml)
for k, s in S.items():
    cfg = dict(rMod=s['rMod'], n_subap=s['n_subap'], wavelength_nm=s['wavelength_nm'])
    fig = pl.plot_calibration(m, 'og', hue='seeing', values=[0.6, 0.75, 0.9, 1.05], **cfg, logx=True, ylim=[0, 1],
                              title=f"{k}: rMod {s['rMod']}, {s['wavelength_nm']:.0f} nm")
    fig = pl.plot_calibration(m, 'noise', hue='flux', values=[s['flux'] * 10.0 ** -i for i in range(4)], **cfg, fs=s['fs'],
                              seeing=e2e.snap_seeing(seeing), logx=True, title=f"{k}: rMod {s['rMod']}, {s['fs']:.0f} Hz")
# fig = pl.plot_calibration(m, 'thrp', hue='rMod', values=[0, 0.5, 1, 2, 3], n_subap=48, seeing=0.9, wavelength_nm=900)
# fig = pl.plot_modal_variance_check(m, rMod=3, n_subap=48, seeing=0.9, wavelength_nm=900)

In [ ]:
eng = PsfEngine(m)                   # caches the KL kernels and the telescope OTF
PS, RES = {}, {}
for k, r in R.items():
    PS[k] = eng.psd_terms(r)           # lag, noise_photon, noise_ron, alias, xtalk, fitting, chrom_shear, total, turbulence (open loop)
    RES[k] = eng.psf_terms(r, S[k]['science_nm'])     # long-exposure PSF of every term alone and of the total
    pl.plot_psd_terms(PS[k], title=f'{k} pyramid')
    pl.plot_psf_image(RES[k], title=f"{k}: total at {S[k]['science_nm']:.0f} nm, Strehl {RES[k].strehl['total']:.3f}")
    pl.plot_psf_terms(RES[k], rmax=25, title=f'{k} pyramid')
plt.show()
pd.DataFrame({k: PS[k].variances() for k in PS}).pow(0.5).round(2)    # [nm rms] of each PSD term (check against the budget table)

In [ ]:
# Raw contrast after a perfect coronagraph (e2e metric: 2-5 lambda/D in the science band of the sensor)
rho = [2, 3, 4, 5, 10, 20]
for k in RES:
    pl.plot_psf_terms(RES[k], kind='coro', rmax=25, title=f'{k} pyramid')
plt.show()
pd.DataFrame({f"{k} ({S[k]['science_nm']:.0f} nm)": RES[k].contrast_at(rho, kind='coro') for k in RES},
             index=pd.Index(rho, name='lambda/D'))

In [ ]:
# Scan around the e2e system (single stage; 'multistage' is not calibrated in e2e). About 10 min: set RUN_SCAN = False to reuse the csv
import os
RUN_SCAN = not os.path.exists('results/e2e_scan_totals.csv')
if RUN_SCAN:
    modes, totals = m.scan(seeings=[0.6, 0.75, 0.9, 1.05], rMods=[0.0, 1.0, 2.0, 3.0], n_subaps=[48],
                           fluxes=(10.0 ** np.arange(7, 11)).tolist(), freqs=[500., 1000., 2000.], scenarios=('standard',),
                           controllers=(CONTROLLER,), wavelengths=[S['vis']['wavelength_nm'], S['nir']['wavelength_nm']],
                           store_modes=list(range(N_dm)), latency_s=latency_s, crosstalk=True, optimize_freq=False)
    os.makedirs('results', exist_ok=True)
    totals.to_csv('results/e2e_scan_totals.csv', index=False)
    modes.to_csv('results/e2e_scan_modes.csv', index=False)
    print(totals.shape, modes.shape)

In [ ]:
totals = pd.read_csv('results/e2e_scan_totals.csv')
modes = pd.read_csv('results/e2e_scan_modes.csv')
v = S['vis']
one = {'scenario': 'standard', 'controller': CONTROLLER, 'rMod': v['rMod'], 'seeing': 0.75, 'Nsubaps': v['n_subap'],
       'wavelength_nm': v['wavelength_nm']}

# 1. every term of ONE configuration (the VIS pyramid of ristretto_main) versus flux
fig = pl.plot_budget_terms(totals, x='photPerMs', filters={**one, 'freqInHz': v['fs']},
                           terms=('lag', 'noise_photon', 'noise_ron', 'alias', 'xtalk', 'fitting', 'total'))
# fig.savefig('terms_vs_flux.png', dpi=150)

# 2. one quantity, a curve per frame rate, a panel per sensing wavelength
fig = pl.plot_budget_compare(totals, x='photPerMs', hue='freqInHz', quantity='total', col='wavelength_nm',
                             filters={k: x for k, x in one.items() if k != 'wavelength_nm'})

# 3. per-mode residual of one configuration, a curve per flux
fig = pl.plot_per_mode(modes, y='residual', hue='photPerMs', filters={**one, 'freqInHz': v['fs']})
plt.show()